# PRO130 – Recepción de Materiales y repuestos (2° versión – Noviembre 2021)
Checklist/HMI piloto (on-site): Recepción física → (Revisión secundaria / Inspección técnica) → Disponibilización en almacén **o** Devolución a proveedores.

**Transacciones SAP (según PRO130):**
- MIGO 101: Ingreso mercadería (109 en importaciones)
- MIGO 103: Ingresar stock bloqueado
- MIGO 105: Liberar/Quitar de stock bloqueado a libre utilización (109 en importaciones, según corresponda)
- MIGO 122: Devolución
- ZMM_MANT_CARACT / ZMM_IMP_ETIQUETA: Características / impresión de etiqueta

**Nota de diseño HMI:** Se agregó un paso explícito **"Contactar a Especialista de Mantenimiento"** entre MIGO 103 y el rombo de decisión de inspección, para trazabilidad (evidencia + timestamp).


In [1]:
import ipywidgets as widgets
from IPython.display import display
import datetime, json, uuid

EN_CURSO = "EN_CURSO"
BLOQUEADO = "BLOQUEADO"
DETENIDO_STOP = "DETENIDO_STOP"
FINALIZADO = "FINALIZADO"
DERIVADO_PROVEEDORES = "DERIVADO_DEVOLUCION_PROVEEDORES"

# -------------------------
# PRO130 – Recepción de Materiales y repuestos (2° versión – Nov-2021)
# Enfoque on-site: Especialista de Almacén + Especialista de Mantenimiento (cuando aplica inspección técnica)
# -------------------------

NODOS = {
    # -----------------------------
    # 0) Recepción física
    # -----------------------------
    "T1_recibir_y_revisar_documentacion": {
        "type": "task",
        "titulo": "Recibir y revisar documentación",
        "rol": "Especialista de Almacén",
        "descripcion": "Recepción física del material/repuesto. Revisar documentación asociada y contrastar con lo solicitado.",
        "acciones": [
            "Recibir material (proveniente de compra de materiales y repuestos, o desde otro centro/producto de venta intercompany).",
            "Revisar documentación de recepción asociada al despacho (guía/packing list/factura u otros respaldos disponibles).",
            "Contrastar material y documentación con lo solicitado (OC/SOLPED/ítems)."
        ],
        "validacion": "¿Se revisó la documentación y se contrastó el material con lo solicitado?",
        "next": "D1_es_lo_que_se_pidio",
    },

    "D1_es_lo_que_se_pidio": {
        "type": "decision",
        "titulo": "¿Es lo que se pidió?",
        "rol": "Especialista de Almacén",
        "pregunta": "¿El material recibido corresponde a lo que se pidió?",
        "opciones": [
            {"label": "SÍ → Evaluar si requiere inspección técnica", "next": "D2_necesita_inspeccion_tecnica"},
            {"label": "NO → Evaluar si requiere inspección técnica (posible devolución inmediata)", "next": "D2b_necesita_inspeccion_tecnica_no_es_pedido"}
        ],
        "ayuda": "Según PRO130: si NO es lo pedido, se define si requiere inspección técnica; si NO requiere, se devuelve de inmediato (se une a devolución a proveedores).",
    },

    # -----------------------------
    # 1) Si NO es lo pedido: decidir inspección técnica
    # -----------------------------
    "D2b_necesita_inspeccion_tecnica_no_es_pedido": {
        "type": "decision",
        "titulo": "¿Se necesita inspección técnica? (NO es lo pedido)",
        "rol": "Especialista de Almacén",
        "pregunta": "Dado que NO corresponde a lo pedido: ¿se necesita inspección técnica para decidir aceptación/rechazo?",
        "opciones": [
            {"label": "SÍ → Ingresar como stock bloqueado (MIGO 103) y contactar mantenimiento", "next": "T3_ingresar_stock_bloqueado_103"},
            {"label": "NO → Devolver de inmediato y derivar a devolución a proveedores", "next": "T2_devolver_de_inmediato"},
        ],
        "ayuda": "Casos típicos que requieren inspección técnica (PRO130): conexiones no estándar, primer ingreso, fabricación/maestranza, daño, homologación/actualización N° parte, materiales reparables.",
    },

    "T2_devolver_de_inmediato": {
        "type": "task",
        "titulo": "Devolver de inmediato",
        "rol": "Especialista de Almacén",
        "descripcion": "Cuando el material NO es lo pedido y NO requiere inspección técnica, la mercadería puede ser devuelta inmediatamente al proveedor, uniéndose al proceso de devolución.",
        "acciones": [
            "Gestionar devolución inmediata del material (según práctica local y acuerdos).",
            "Registrar devolución en SAP (movimiento 122).",
            "Dejar evidencia de la devolución (documentación asociada y comunicación)"
        ],
        "validacion": "¿La devolución inmediata quedó gestionada y registrada en SAP (MIGO 122) con evidencia?",
        "next": "END_derivar_devolucion_proveedores",
    },

    # -----------------------------
    # 2) Si ES lo pedido: decidir inspección técnica
    # -----------------------------
    "D2_necesita_inspeccion_tecnica": {
        "type": "decision",
        "titulo": "¿Se necesita inspección técnica?",
        "rol": "Especialista de Almacén",
        "pregunta": "¿El material requiere inspección técnica?",
        "opciones": [
            {"label": "SÍ → Ingresar como stock bloqueado (MIGO 103) y contactar mantenimiento", "next": "T3_ingresar_stock_bloqueado_103"},
            {"label": "NO → Ingresar recepción (MIGO 101 / 109 importaciones) y revisión secundaria", "next": "T4_ingresar_recepcion_101"},
        ],
        "ayuda": "PRO130: requiere inspección técnica en casos como conexiones no estándar, primer ingreso, fabricación/maestranza, daño, homologación/actualización, materiales reparables.",
    },

    # -----------------------------
    # 3) Ruta SIN inspección técnica: ingreso + revisión secundaria
    # -----------------------------
    "T4_ingresar_recepcion_101": {
        "type": "task",
        "titulo": "Ingresar recepción (MIGO)",
        "rol": "Especialista de Almacén",
        "descripcion": "Si NO requiere inspección técnica, ingresar mercadería por MIGO con movimiento 101 (109 en importaciones) y someter a revisión secundaria pasiva.",
        "acciones": [
            "Ingresar mercadería en SAP mediante transacción MIGO (movimiento 101; 109 en importaciones).",
            "Ejecutar 'revisión secundaria' de manera pasiva (según práctica local) para detectar diferencias posteriores."
        ],
        "validacion": "¿La recepción fue ingresada en MIGO (101/109) y quedó iniciada la revisión secundaria?",
        "next": "T5_realizar_revision_secundaria",
    },

    "T5_realizar_revision_secundaria": {
        "type": "task",
        "titulo": "Realizar inspección secundaria",
        "rol": "Especialista de Almacén",
        "descripcion": "Revisión secundaria pasiva para confirmar que no existan diferencias con lo pedido tras el ingreso.",
        "acciones": [
            "Completar revisión secundaria pasiva del material/documentación.",
            "Identificar si existen diferencias con lo pedido (cantidad, identificación, estado, etc.)."
        ],
        "validacion": "¿La revisión secundaria se completó y se determinó si existen diferencias con lo pedido?",
        "next": "D3_es_lo_que_se_pidio_post_revision",
    },

    "D3_es_lo_que_se_pidio_post_revision": {
        "type": "decision",
        "titulo": "¿Es lo que se pidió? (post revisión secundaria)",
        "rol": "Especialista de Almacén",
        "pregunta": "Tras la revisión secundaria: ¿el material coincide con lo pedido (sin diferencias)?",
        "opciones": [
            {"label": "SÍ → Gestionar disponibilización en almacén (ubicación + etiqueta)", "next": "T10_determinar_ubicacion_y_datos_etiquetado"},
            {"label": "NO → Quitar stock del sistema y derivar a devolución a proveedores", "next": "T9_quitar_stock_del_sistema"},
        ],
        "ayuda": "En el diagrama PRO130: si hay diferencias post-revisión, se quita stock y se deriva a devolución a proveedores.",
    },

    # -----------------------------
    # 4) Ruta CON inspección técnica: stock bloqueado + contacto + decisión/inspección
    # -----------------------------
    "T3_ingresar_stock_bloqueado_103": {
        "type": "task",
        "titulo": "Ingresar recepción como stock bloqueado (MIGO 103)",
        "rol": "Especialista de Almacén",
        "descripcion": "Cuando se requiere inspección técnica, ingresar la mercadería como stock bloqueado mediante movimiento 103.",
        "acciones": [
            "Ingresar mercadería como stock bloqueado en SAP mediante transacción MIGO (movimiento 103).",
            "Verificar en SAP que el stock quedó efectivamente en condición 'bloqueado'."
        ],
        "validacion": "¿El material quedó ingresado como stock bloqueado (MIGO 103) y verificado en SAP?",
        "next": "T3b_contactar_mantencion",
    },

    "T3b_contactar_mantencion": {
        "type": "task",
        "titulo": "Contactar a Especialista de Mantenimiento",
        "rol": "Especialista de Almacén",
        "descripcion": "Notificar formalmente a Mantenimiento para que evalúe si realizará inspección técnica. PRO130 considera plazo de 48 h hábiles desde la notificación para ejecutar la inspección (cuando aplique).",
        "acciones": [
            "Contactar al Especialista de Mantenimiento informando: material/repuesto, motivo de inspección, y ubicación temporal/stock bloqueado.",
            "Adjuntar/entregar respaldos disponibles (OC/guía, fotos si aplica, antecedentes técnicos).",
            "Registrar fecha/hora de notificación (timestamp) y dejar evidencia (correo/registro)."
        ],
        "validacion": "¿Mantenimiento fue contactado y quedó evidencia + timestamp de la notificación (para control de 48 h hábiles)?",
        "next": "D4_mant_realiza_inspeccion",
    },

    "D4_mant_realiza_inspeccion": {
        "type": "decision",
        "titulo": "¿Especialista realiza inspección?",
        "rol": "Especialista de Mantenimiento",
        "pregunta": "¿El especialista de mantenimiento realizará inspección técnica?",
        "opciones": [
            {"label": "NO → Liberar a stock libre utilización (MIGO 105 / 109 importaciones)", "next": "T6_traspasar_a_libre_utilizacion_105"},
            {"label": "SÍ → Realizar inspección (plazo 48 h hábiles) y decidir aprobación", "next": "T7_realizar_inspeccion_mant"},
        ],
        "ayuda": "PRO130: si NO inspecciona, se ejecuta 105 (109 importaciones). Si SÍ inspecciona, tiene 48 h hábiles desde notificación para realizarla.",
    },

    "T7_realizar_inspeccion_mant": {
        "type": "task",
        "titulo": "Realizar inspección (plazo 48 h hábiles)",
        "rol": "Especialista de Mantenimiento",
        "descripcion": "Inspección técnica del material/repuesto requerido. PRO130 indica 48 horas hábiles desde la notificación para realizar la inspección.",
        "acciones": [
            "Realizar inspección técnica del material según necesidad (condición, compatibilidad, homologación, daño, etc.).",
            "Registrar resultado/observaciones de inspección (según práctica local)."
        ],
        "validacion": "¿La inspección técnica fue realizada dentro del plazo (48 h hábiles desde notificación) y se registró el resultado?",
        "next": "D5_aprueba_material",
    },

    "D5_aprueba_material": {
        "type": "decision",
        "titulo": "¿Se aprueba el material?",
        "rol": "Especialista de Mantenimiento",
        "pregunta": "¿El material queda aprobado para uso (aceptación técnica)?",
        "opciones": [
            {"label": "SÍ → Traspasar a stock libre utilización (MIGO 105 / 109 importaciones)", "next": "T6_traspasar_a_libre_utilizacion_105"},
            {"label": "NO → Traspasar a stock para comenzar devolución y quitar stock", "next": "T8_traspasar_stock_devolucion"},
        ],
        "ayuda": "En el diagrama PRO130: si NO se aprueba, se deriva a devolución a proveedores (movimiento 122).",
    },

    "T6_traspasar_a_libre_utilizacion_105": {
        "type": "task",
        "titulo": "Traspasar material/repuesto a stock de libre utilización (MIGO 105)",
        "rol": "Especialista de Almacén",
        "descripcion": "Liberar el material desde stock bloqueado hacia libre utilización mediante MIGO 105 (109 en importaciones), luego continúa a gestión de disponibilización en almacén.",
        "acciones": [
            "Ejecutar traspaso a stock de libre utilización en SAP mediante transacción MIGO (movimiento 105; 109 en importaciones si aplica).",
            "Verificar que el stock quedó disponible para continuar con ubicación y etiquetado."
        ],
        "validacion": "¿El material fue liberado a stock de libre utilización (MIGO 105/109) y quedó disponible para ubicación/etiquetado?",
        "next": "T10_determinar_ubicacion_y_datos_etiquetado",
    },

    "T8_traspasar_stock_devolucion": {
        "type": "task",
        "titulo": "Traspasar material/repuesto a stock para comenzar devolución",
        "rol": "Especialista de Almacén",
        "descripcion": "Cuando el material NO se aprueba, traspasar a stock para comenzar devolución y preparar antecedentes.",
        "acciones": [
            "Traspasar material/repuesto a stock para comenzar devolución (según práctica local).",
            "Preparar antecedentes/documentación para devolución al proveedor."
        ],
        "validacion": "¿El material quedó preparado/traspasado para iniciar devolución y con antecedentes listos?",
        "next": "T9_quitar_stock_del_sistema",
    },

    "T9_quitar_stock_del_sistema": {
        "type": "task",
        "titulo": "Quitar stock del sistema (previo a devolución)",
        "rol": "Especialista de Almacén",
        "descripcion": "Quitar stock del sistema antes de derivar a la devolución a proveedores (según flujo del PRO130).",
        "acciones": [
            "Ejecutar la acción correspondiente para quitar stock del sistema (según transacción/movimiento definido localmente).",
            "Asegurar que el flujo queda listo para devolución a proveedores (movimiento 122)."
        ],
        "validacion": "¿El stock fue retirado del sistema y el material quedó listo para devolución (MIGO 122)?",
        "next": "END_derivar_devolucion_proveedores",
    },

    "END_derivar_devolucion_proveedores": {
        "type": "end",
        "titulo": "Derivar a Devolución a proveedores (MIGO 122)",
        "rol": "Especialista de Almacén",
        "mensaje": "Continuar con el proceso de Devolución a proveedores ejecutando devolución en MIGO con movimiento 122 (según PRO130).",
        "estado_final": DERIVADO_PROVEEDORES,
    },

    # -----------------------------
    # 5) Gestión para disponibilización en almacén (ubicación + etiqueta)
    # -----------------------------
    "T10_determinar_ubicacion_y_datos_etiquetado": {
        "type": "task",
        "titulo": "Determinar ubicación y verificar datos para etiquetado",
        "rol": "Especialista de Almacén",
        "descripcion": "Gestionar disponibilización en almacén: determinar ubicación, verificar datos para etiquetado y mantener características en sistema.",
        "acciones": [
            "Determinar ubicación del material en almacén.",
            "Verificar datos necesarios para etiquetado y mantener características (ZMM_MANT_CARACT).",
        ],
        "validacion": "¿Se determinó la ubicación y se verificaron/mantuvieron datos para etiquetado (ZMM_MANT_CARACT)?",
        "next": "T11_imprimir_etiqueta",
    },

    "T11_imprimir_etiqueta": {
        "type": "task",
        "titulo": "Imprimir etiqueta (ZMM_IMP_ETIQUETA)",
        "rol": "Especialista de Almacén",
        "descripcion": "Imprimir etiqueta en SAP para identificación del material/repuesto.",
        "inputs": [
            {"key": "numero_etiqueta", "label": "Ingresa número de etiqueta", "required": True}
        ],

        "acciones": [
            "Imprimir etiqueta mediante transacción ZMM_IMP_ETIQUETA.",
            "Verificar que la etiqueta corresponde al material y contiene información correcta."
        ],
        "validacion": "¿La etiqueta fue impresa correctamente y corresponde al material?",
        "next": "T12_disponer_material_en_ubicacion",
    },

    "T12_disponer_material_en_ubicacion": {
        "type": "task",
        "titulo": "Disponer material en la ubicación asignada",
        "rol": "Especialista de Almacén",
        "descripcion": "Ubicar físicamente el material en la ubicación asignada y dejarlo disponible.",
        "acciones": [
            "Ubicar físicamente el material/repuesto en la ubicación asignada.",
            "Verificar que el material queda ordenado y accesible según estándar local."
        ],
        "validacion": "¿El material quedó dispuesto físicamente en la ubicación asignada?",
        "next": "T13_notificar_disponibilidad_materiales",
    },

    "T13_notificar_disponibilidad_materiales": {
        "type": "task",
        "titulo": "Notificar disponibilidad de materiales",
        "rol": "Especialista de Almacén",
        "descripcion": "Notificar disponibilidad de materiales una vez almacenados y disponibles.",
        "acciones": [
            "Notificar a quienes corresponda que el material quedó disponible en almacén (según práctica local).",
        ],
        "validacion": "¿Se notificó la disponibilidad del material y quedó evidencia de la notificación?",
        "next": "END_material_almacenado",
    },

    "END_material_almacenado": {
        "type": "end",
        "titulo": "Fin: Material almacenado",
        "rol": "Especialista de Almacén",
        "mensaje": "Material recepcionado, validado y disponibilizado en almacén según PRO130.",
        "estado_final": FINALIZADO,
    },
}

# -------------------------
# Motor HMI (misma potencia que el piloto: bloqueo + rehacer + volver + export JSON)
# -------------------------

def _now_iso():
    return datetime.datetime.now().isoformat(timespec="seconds")

class PRO130HMI:
    def __init__(self):
        self.nodo_id = "T1_recibir_y_revisar_documentacion"
        self.historial = []  # stack de nodos visitados
        self.logs = []       # eventos para exportar
        self.output = widgets.Output(layout={"width":"100%"})

        # UI controls
        self.btn_si = widgets.Button(description="SÍ", button_style="success", layout={"width":"48%","height":"44px"})
        self.btn_no = widgets.Button(description="NO", button_style="danger", layout={"width":"48%","height":"44px"})
        self.btn_volver = widgets.Button(description="⬅ Volver al paso anterior", layout={"width":"100%","height":"40px"})
        self.btn_exportar = widgets.Button(description="Exportar JSON (trazabilidad)", icon="download", layout={"width":"100%","height":"40px"})

        self.msg_box = widgets.HTML("")
        self.main_box = widgets.VBox([])

        # Bloqueos (motivo + rehacer)
        self.is_blocked = False
        self.block_reason = None
        self.block_panel = widgets.VBox([])
        self.btn_rehacer = widgets.Button(description="🔄 Rehacer paso", button_style="info", layout={"width":"100%","height":"40px"})
        self.btn_rehacer.on_click(self._on_rehacer)

        self._wire()
        self._render()

    def _wire(self):
        self.btn_si.on_click(self._on_si)
        self.btn_no.on_click(self._on_no)
        self.btn_volver.on_click(self._on_volver)
        self.btn_exportar.on_click(self._on_exportar)

    def _log(self, tipo, data=None):
        self.logs.append({
            "ts": _now_iso(),
            "tipo": tipo,
            "nodo_id": self.nodo_id,
            "data": data or {}
        })

    def _push_hist(self, prev_id):
        self.historial.append(prev_id)

    def _pop_hist(self):
        if self.historial:
            return self.historial.pop()
        return None

    def _set_msg(self, html):
        self.msg_box.value = html

    def _clear_msg(self):
        self.msg_box.value = ""

    def _render_header(self, n):
        badge = f"<span style='display:inline-block;padding:4px 10px;border-radius:999px;background:#eef2ff;border:1px solid #c7d2fe;font-size:12px;color:black;'><b>ROL:</b> {n.get('rol','')}</span>"
        return widgets.HTML(f"""
        <div style="padding:16px;border-radius:12px;background:#f8fafc;border:1px solid #e2e8f0;">
            <div style="font-size:12px;color:#0f172a;"><b>PRO130</b> – Recepción de Materiales y repuestos (2° versión – Nov-2021)</div>
            <div style="margin-top:6px;font-size:22px;color:#0f172a;"><b>{n.get('titulo','')}</b></div>
            <div style="margin-top:8px;">{badge}</div>
            <div style="margin-top:10px;color:#0f172a;font-size:14px;line-height:1.35;">{n.get('descripcion','')}</div>
        </div>
        """)

    def _render_task(self, n):
        # Guarda widgets para validación
        self._task_checkboxes = []
        self._task_inputs = {}

        # Checklist (acciones) -> checkboxes
        acciones = n.get("acciones", []) or []
        cb_widgets = []
        for a in acciones:
            cb = widgets.Checkbox(
                value=False,
                description=a,
                indent=False,
                layout=widgets.Layout(width="100%")
            )
            self._task_checkboxes.append((a, cb))
            cb_widgets.append(cb)

        # Inputs del paso (opcional)
        input_widgets = []
        for inp in (n.get("inputs") or []):
            key = inp.get("key")
            label = inp.get("label", key or "")
            required = bool(inp.get("required", False))
            multiline = bool(inp.get("multiline", False))

            if multiline:
                w = widgets.Textarea(
                    value="",
                    placeholder="",
                    layout=widgets.Layout(width="100%")
                )
            else:
                w = widgets.Text(
                    value="",
                    placeholder="",
                    layout=widgets.Layout(width="100%")
                )

            self._task_inputs[key] = {"widget": w, "label": label, "required": required}
            # Label tipo PRO (sin cambiar estilo global)
            req_tag = " <span style='color:#ef4444;font-weight:700;'>(*)</span>" if required else ""
            input_widgets.append(widgets.HTML(f"<div style='margin-top:10px;font-size:13px;color:#0f172a;'><b>{label}</b>{req_tag}</div>"))
            input_widgets.append(w)

        valid = n.get("validacion","")

        # Card acciones + checklist
        acciones_card = widgets.VBox(
            [
                widgets.HTML("<div style='font-size:13px;color:#0f172a;'><b>⚙️ ACCIÓN A EJECUTAR (texto PRO130)</b></div>"),
                widgets.VBox(cb_widgets, layout=widgets.Layout(margin="10px 0 0 0")),
                *input_widgets
            ],
            layout=widgets.Layout(
                margin="12px 0 0 0",
                padding="14px",
                border="1px solid #e2e8f0",
                border_radius="12px",
                width="100%"
            )
        )

        valid_card = widgets.HTML(f"""
        <div style="margin-top:12px;padding:14px;border-radius:12px;border:2px solid #0ea5e9;background:#ffffff;">
            <div style="font-size:13px;color:#0f172a;"><b>✅ ¡VALIDACIÓN!</b></div>
            <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{valid}</b></div>
            <div style="margin-top:6px;font-size:12px;color:#0f172a;">Confirma con <b>SÍ</b> para avanzar. Si respondes <b>NO</b>, el paso queda bloqueado.</div>
        </div>
        """)

        return widgets.VBox([acciones_card, valid_card])

    def _render_decision(self, n):
        opts = n.get("opciones",[])
        radios = widgets.RadioButtons(
            options=[(o["label"], o["next"]) for o in opts],
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )
        help_txt = n.get("ayuda","")
        help_html = f"<div style='margin-top:10px;font-size:12px;color:#0f172a;opacity:0.9;'><b>Nota:</b> {help_txt}</div>" if help_txt else ""
        return widgets.VBox([
            widgets.HTML(f"""
            <div style="margin-top:12px;padding:14px;border-radius:12px;border:1px solid #e2e8f0;background:#ffffff;">
                <div style="font-size:13px;color:#0f172a;"><b>🔶 DECISIÓN (rombo)</b></div>
                <div style="margin-top:8px;font-size:16px;color:#0f172a;"><b>{n.get('pregunta','')}</b></div>
                {help_html}
            </div>
            """),
            radios
        ]), radios

    def _render_footer(self):
        self.btn_volver.disabled = (len(self.historial) == 0)
        return widgets.VBox([
            widgets.HBox([self.btn_si, self.btn_no], layout={"justify_content":"space-between","margin":"10px 0"}),
            self.btn_volver,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.btn_exportar,
            widgets.HTML("<div style='height:10px;'></div>"),
            self.block_panel,
            self.msg_box,
        ])

    def _render(self):
        with self.output:
            self.output.clear_output()

            n = NODOS[self.nodo_id]
            header = self._render_header(n)

            if n["type"] == "task":
                body = self._render_task(n)
                self._decision_widget = None
            elif n["type"] == "decision":
                body, radios = self._render_decision(n)
                self._decision_widget = radios
            elif n["type"] == "end":
                self._decision_widget = None
                body = widgets.HTML(f"""
                <div style="margin-top:12px;padding:18px;border-radius:12px;border:2px solid #22c55e;background:#f0fdf4;">
                    <div style="font-size:20px;color:#0f172a;"><b>🏁 FIN / DERIVACIÓN</b></div>
                    <div style="margin-top:10px;font-size:15px;color:#0f172a;">{n.get('mensaje','')}</div>
                    <div style="margin-top:10px;font-size:12px;color:#0f172a;">Estado final: <b>{n.get('estado_final','')}</b></div>
                </div>
                """)
            else:
                body = widgets.HTML("<div>Tipo de nodo no soportado.</div>")

            footer = self._render_footer()
            self.main_box.children = [header, body, footer]
            display(self.main_box)

    def _advance_to(self, next_id):
        prev = self.nodo_id
        self._push_hist(prev)
        self.nodo_id = next_id
        self._log("AVANZA", {"from": prev, "to": next_id})
        self._clear_msg()
        self._render()

    def _on_si(self, _):
        if getattr(self, "is_blocked", False):
            self._set_msg("""<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
                <b>⛔ Paso bloqueado:</b> Primero registra el motivo y usa <b>Rehacer paso</b> para volver a ejecutar/corregir. Luego valida con <b>SÍ</b>.
            </div>""")
            return
        n = NODOS[self.nodo_id]
        if n["type"] == "end":
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Ya estás en un fin/derivación.</div>")
            return

        if n["type"] == "task":
            # Validación de checklist (bloquea avance si falta alguna acción obligatoria)
            faltantes = []
            for txt, cb in getattr(self, "_task_checkboxes", []):
                if ("si aplica" in (txt or "").lower()):
                    continue
                if not cb.value:
                    faltantes.append(txt)

            # Validación de inputs requeridos (si aplica)
            faltan_inputs = []
            for key, meta in getattr(self, "_task_inputs", {}).items():
                w = meta["widget"]
                required = bool(meta.get("required", False))
                label = meta.get("label", key)
                if required and (str(w.value).strip() == ""):
                    faltan_inputs.append(label)

            if faltantes or faltan_inputs:
                items_html = ""
                if faltantes:
                    items_html += "<div style='margin-top:6px;'><b>Checklist pendiente:</b><ul style='margin:6px 0 0 18px;'>" + "".join([f"<li>{x}</li>" for x in faltantes]) + "</ul></div>"
                if faltan_inputs:
                    items_html += "<div style='margin-top:6px;'><b>Campos requeridos:</b><ul style='margin:6px 0 0 18px;'>" + "".join([f"<li>{x}</li>" for x in faltan_inputs]) + "</ul></div>"
                self._set_msg(f"""<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
                    <b>Falta completar el paso:</b> Marca todas las acciones obligatorias y completa los campos requeridos para avanzar.
                    {items_html}
                </div>""")
                return

            # Log de completitud del paso
            data = {
                "acciones": [{"texto": t, "checked": bool(cb.value)} for t, cb in getattr(self, "_task_checkboxes", [])],
                "inputs": {k: str(v["widget"].value) for k, v in getattr(self, "_task_inputs", {}).items()},
            }
            self._log("TASK_OK", data)

            next_id = n.get("next")
            if next_id:
                self._advance_to(next_id)
            else:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fff7ed;border:1px solid #fdba74;color:#0f172a;'><b>Atención:</b> Este paso no tiene siguiente definido.</div>")
            return


        if n["type"] == "decision":
            if self._decision_widget is None or self._decision_widget.value is None:
                self._set_msg("<div style='padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta selección:</b> Elige una opción para avanzar.</div>")
                return
            self._advance_to(self._decision_widget.value)
            return

    def _motivos_bloqueo(self):
        nid = self.nodo_id
        if nid in ("T1_recibir_y_revisar_documentacion", "D1_es_lo_que_se_pidio"):
            return [
                "Falta documentación de recepción (guía/factura/packing u otros)",
                "Material no coincide con ítem solicitado (identificación/cantidad)",
                "Riesgo de seguridad (sustancias peligrosas / daño visible) – detener y gestionar",
            ]
        if nid in ("D2_necesita_inspeccion_tecnica", "D2b_necesita_inspeccion_tecnica_no_es_pedido"):
            return [
                "Caso requiere inspección técnica (conexión no estándar / primer ingreso / homologación / daño / reparable / maestranza)",
                "No hay claridad para aceptar/rechazar sin inspección",
            ]
        if nid in ("T4_ingresar_recepcion_101","T5_realizar_revision_secundaria","D3_es_lo_que_se_pidio_post_revision"):
            return [
                "Error al registrar ingreso en SAP (MIGO 101/109)",
                "Diferencias detectadas en revisión secundaria",
            ]
        if nid in ("T3_ingresar_stock_bloqueado_103","T3b_contactar_mantencion"):
            return [
                "Error al registrar stock bloqueado (MIGO 103)",
                "No se logró contactar a mantenimiento / falta confirmación de recepción del aviso",
                "Falta evidencia/timestamp de notificación a mantenimiento",
            ]
        if nid in ("D4_mant_realiza_inspeccion","T7_realizar_inspeccion_mant","D5_aprueba_material"):
            return [
                "Mantenimiento no responde / inspección pendiente (48 h hábiles)",
                "Resultado de inspección no registrado / falta evidencia técnica",
                "Material rechazado por inspección técnica – preparar devolución",
            ]
        if nid in ("T6_traspasar_a_libre_utilizacion_105","T10_determinar_ubicacion_y_datos_etiquetado","T11_imprimir_etiqueta","T12_disponer_material_en_ubicacion","T13_notificar_disponibilidad_materiales"):
            return [
                "Error al liberar stock (MIGO 105/109)",
                "No se puede determinar ubicación / datos incompletos para etiqueta",
                "Etiqueta impresa incorrecta / datos erróneos",
                "No se pudo notificar disponibilidad",
            ]
        if nid in ("T9_quitar_stock_del_sistema","T8_traspasar_stock_devolucion","T2_devolver_de_inmediato"):
            return [
                "No se pudo preparar/registrar devolución (MIGO 122)",
                "Falta evidencia/documentación para devolución",
            ]
        return [
            "Pendiente validación/confirmación del responsable del paso",
            "Se requiere información adicional antes de continuar",
        ]

    def _on_no(self, _):
        n = NODOS[self.nodo_id]
        self.is_blocked = True
        self.block_reason = None
        self.btn_si.disabled = True
        self._log("BLOQUEO", {"titulo": n.get("titulo","")})

        motivos = self._motivos_bloqueo()
        radio = widgets.RadioButtons(
            options=motivos,
            layout={"width":"100%"},
            style={"description_width":"initial"},
        )

        def _on_pick(change):
            if change.get("name") == "value":
                self.block_reason = change["new"]
                self._log("MOTIVO_BLOQUEO_SELECCIONADO", {"motivo": self.block_reason})
                self._set_msg("""
                <div style='margin-top:10px;padding:10px;border-radius:10px;background:#e0f2fe;border:1px solid #0284c7;color:#0f172a;'>
                    <b>✅ Motivo registrado.</b> Ahora ejecuta la corrección indicada y presiona <b>Rehacer paso</b>.
                </div>
                """)

        radio.observe(_on_pick, names="value")

        self.block_panel.children = [
            widgets.HTML("""
            <div style='margin-top:10px;padding:12px;border-radius:12px;background:#fee2e2;border:1px solid #ef4444;color:#0f172a;'>
                <b>🛑 BLOQUEADO:</b> Debes indicar el motivo (quedará en la trazabilidad) y luego rehacer el paso.
            </div>
            """),
            widgets.HTML("<div style='margin-top:6px;font-size:13px;color:#0f172a;'><b>Motivo del bloqueo (según PRO130):</b></div>"),
            radio,
            widgets.HTML("<div style='height:8px;'></div>"),
            self.btn_rehacer,
        ]

        self._set_msg("""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'>
            <b>Instrucción:</b> Selecciona un motivo, realiza la corrección/gestión y luego presiona <b>Rehacer paso</b>.
        </div>
        """)

    def _on_rehacer(self, _):
        if not getattr(self, "is_blocked", False):
            self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Este paso no está bloqueado.</div>")
            return
        if not self.block_reason:
            self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#fffbeb;border:1px solid #f59e0b;color:#0f172a;'><b>Falta motivo:</b> Debes seleccionar un motivo de bloqueo antes de rehacer.</div>")
            return

        self._log("REHACER_PASO", {"motivo": self.block_reason})
        self.is_blocked = False
        self.btn_si.disabled = False
        self.block_panel.children = []
        self._set_msg("<div style='margin-top:10px;padding:12px;border-radius:10px;background:#e8f5e9;border:1px solid #22c55e;color:#0f172a;'><b>🔄 Paso listo para rehacer.</b> Ejecuta la acción del paso actual y valida con <b>SÍ</b>.</div>")

    def _on_volver(self, _):
        prev = self._pop_hist()
        # Al volver, limpiar bloqueo UI
        self.is_blocked = False
        self.block_reason = None
        self.block_panel.children = []
        self.btn_si.disabled = False
        if prev is None:
            self._set_msg("<div style='padding:12px;border-radius:10px;background:#e2e8f0;border:1px solid #cbd5e1;color:#0f172a;'><b>Info:</b> Ya estás en el inicio.</div>")
            return
        cur = self.nodo_id
        self.nodo_id = prev
        self._log("VOLVER", {"from": cur, "to": prev})
        self._clear_msg()
        self._render()

    def _on_exportar(self, _):
        payload = {
            "proceso": "PRO130 – Recepción de Materiales y repuestos (2° versión – Nov-2021)",
            "session_id": str(uuid.uuid4()),
            "export_ts": _now_iso(),
            "current_node": self.nodo_id,
            "history_stack": list(self.historial),
            "logs": list(self.logs),
        }
        pretty = json.dumps(payload, ensure_ascii=False, indent=2)
        self._set_msg(f"""
        <div style='margin-top:10px;padding:12px;border-radius:10px;background:#f1f5f9;border:1px solid #cbd5e1;color:#0f172a;'>
            <b>📦 Export JSON (trazabilidad)</b>
            <pre style='white-space:pre-wrap;margin-top:10px;color:#0f172a;'>{pretty}</pre>
        </div>
        """)

    def iniciar(self):
        display(self.output)

# Iniciar HMI
hmi = PRO130HMI()
hmi.iniciar()


Output(layout=Layout(width='100%'))